In [1]:
import sqlite3
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Connexion à la base
conn = sqlite3.connect('../dev/input/olist.db')

query = """
WITH from_now AS (
    SELECT 
        MAX(order_purchase_timestamp) AS max_timestamp 
    FROM orders o WHERE o.order_status != "canceled"
) 
select  JULIANDAY(n.max_timestamp) - JULIANDAY(o.order_purchase_timestamp) AS recence,
		COUNT(o.order_id) AS frequence,
		oi.price, oi.freight_value, sum(oi.price + oi.freight_value) as montant, rev.review_score, 
    order_purchase_timestamp, order_approved_at, order_delivered_customer_date
    FROM orders o
    CROSS JOIN from_now n
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    INNER JOIN customers c ON o.customer_id = c.customer_id 
    INNER JOIN order_reviews rev ON o.order_id = rev.order_id
    INNER JOIN sellers s ON oi.seller_id = s.seller_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.customer_id
    ORDER BY frequence DESC

    """

df = pd.read_sql(query, conn)

In [2]:
from scipy import stats
import numpy as np

def detect_outliers_zscore(df, column, threshold=3):
    z_scores = np.abs(stats.zscore(df[column]))
    outliers = df[z_scores > threshold]
    return outliers

# Application
numeric_df = df.select_dtypes(include=[np.number])


# Application sur les colonnes numériques uniquement
for column in numeric_df.columns:
    outliers_zscore = detect_outliers_zscore(df, column)
    print(f"Colonne {column} - Outliers Z-score : {len(outliers_zscore)} clients")


Colonne recence - Outliers Z-score : 1 clients
Colonne frequence - Outliers Z-score : 2266 clients
Colonne price - Outliers Z-score : 1692 clients
Colonne freight_value - Outliers Z-score : 1758 clients
Colonne montant - Outliers Z-score : 1703 clients
Colonne review_score - Outliers Z-score : 0 clients


In [3]:
# Détecter les outliers pour les colonnes 'price' et 'freight_value'
outliers_price = detect_outliers_zscore(df, 'price')
outliers_freight = detect_outliers_zscore(df, 'freight_value')

# Combiner les indices des outliers des deux colonnes
outlier_indices = outliers_price.index.union(outliers_freight.index)

# Ensemble de données sans outliers
df_no_outliers = df.drop(outlier_indices)

In [4]:
df_no_outliers['order_purchase_timestamp'] = pd.to_datetime(df_no_outliers['order_purchase_timestamp'])
df_no_outliers['order_approved_at'] = pd.to_datetime(df_no_outliers['order_approved_at'])
df_no_outliers['order_delivered_customer_date'] = pd.to_datetime(df_no_outliers['order_delivered_customer_date'])


# Calcul du délai de validation (en minutes, heures ou jours selon la granularité souhaitée)
df_no_outliers['approval_delay_minutes'] = (df_no_outliers['order_approved_at'] - df_no_outliers['order_purchase_timestamp']).dt.total_seconds() / 60
df_no_outliers['approval_delay_hours'] = df_no_outliers['approval_delay_minutes'] / 60
df_no_outliers['approval_delay_days'] = (df_no_outliers['order_approved_at'] - df_no_outliers['order_purchase_timestamp']).dt.days

# En heures (optionnel)
df_no_outliers['processing_delay_hours'] = (df_no_outliers['order_delivered_customer_date'] - df_no_outliers['order_purchase_timestamp']).dt.total_seconds() / 3600


df_no_outliers['processing_delay_days'] = (df_no_outliers['order_delivered_customer_date'] - df_no_outliers['order_purchase_timestamp']).dt.days

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.stats import ks_2samp

# Charger les données
# Remplacez 'votre_fichier.csv' par le chemin vers votre fichier de données
data = pd.read_csv('votre_fichier.csv')

# Prétraitement des données
data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
data['recence'] = data['recence'].fillna(data['recence'].mean())
data['review_score'] = data['review_score'].fillna(data['review_score'].mean())

# Segmenter les données par période (par exemple, par mois)
data['period'] = data['order_purchase_timestamp'].dt.to_period('3M')
periods = data['period'].unique()

# Fonction pour appliquer K-Means et obtenir les clusters
def apply_kmeans(data, n_clusters=3):
    features = data[['recence', 'frequence', 'montant']]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    return kmeans.fit_predict(features_scaled)

# Appliquer K-Means pour chaque période et stocker les clusters
clusters_periods = [apply_kmeans(data[data['period'] == period]) for period in periods]

# Calculer l'ARI entre les clusters de chaque période et la période initiale
ari_scores = [adjusted_rand_score(clusters_periods[0], clusters) for clusters in clusters_periods]

# Tracer l'évolution de l'ARI
plt.plot(range(len(periods)), ari_scores, marker='o')
plt.title('Évolution de l\'ARI sur différentes périodes')
plt.xlabel('Période')
plt.ylabel('ARI')
plt.xticks(range(len(periods)), [str(period) for period in periods], rotation=45)
plt.show()

# Examiner l'évolution de la distribution des caractéristiques numériques
feature_name = 'montant'
plt.figure(figsize=(12, 6))
for i, period in enumerate(periods):
    sns.kdeplot(data[data['period'] == period][feature_name], label=str(period))
plt.title(f'Évolution de la distribution de {feature_name}')
plt.xlabel(feature_name)
plt.ylabel('Densité')
plt.legend()
plt.show()

# Appliquer le test de Kolmogorov-Smirnov pour comparer les distributions
ks_results = []
for i in range(1, len(periods)):
    ks_stat, p_value = ks_2samp(data[data['period'] == periods[0]][feature_name], data[data['period'] == periods[i]][feature_name])
    ks_results.append((periods[i], ks_stat, p_value))

# Afficher les résultats du test de Kolmogorov-Smirnov
for result in ks_results:
    print(f'Période {result[0]}: KS Statistic={result[1]}, p-value={result[2]}')


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Assurez-vous que la colonne de date est au format datetime
df_no_outliers['order_purchase_timestamp'] = pd.to_datetime(df_no_outliers['order_purchase_timestamp'])

# Créer une colonne pour identifier les périodes de trois mois
df_no_outliers['period'] = df_no_outliers['order_purchase_timestamp'].dt.to_period('3M')

# Sélectionner les caractéristiques pour le clustering
features = ['recence', 'frequence', 'montant', 'review_score']

# Remplacer les valeurs manquantes par la moyenne de chaque colonne
df_no_outliers[features] = df_no_outliers[features].fillna(df_no_outliers[features].mean())

# Fonction pour appliquer K-Means et obtenir les clusters
def apply_kmeans(data, n_clusters=3):
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    
    return kmeans.fit_predict(data_scaled)


# Filtrer les périodes avec suffisamment de données
periods = df_no_outliers['period'].value_counts()[df_no_outliers['period'].value_counts() >= 3].index

# Appliquer K-Means pour chaque période de trois mois
clusters_periods = {}

for period in periods:
    period_data = df_no_outliers[df_no_outliers['period'] == period]
    if len(period_data) > 0:  # Vérifier s'il y a des données pour cette période
        clusters = apply_kmeans(period_data[features])
        print(f"Period {period}: {clusters}")
        clusters_periods[period] = clusters

# Ajouter les clusters au DataFrame
for period in clusters_periods:
    df_no_outliers.loc[df_no_outliers['period'] == period, 'Cluster'] = clusters_periods[period]

# Afficher les résultats
display(df_no_outliers)


Period 2017-11: [2 2 2 ... 0 1 0]
Period 2018-01: [2 2 2 ... 1 1 1]
Period 2018-03: [2 2 2 ... 0 1 1]
Period 2018-04: [0 0 0 ... 1 1 1]
Period 2018-05: [2 2 2 ... 0 0 1]
Period 2018-02: [2 2 2 ... 0 0 0]
Period 2018-08: [2 2 2 ... 1 1 0]
Period 2018-07: [2 2 2 ... 0 0 0]
Period 2018-06: [2 2 2 ... 2 1 1]
Period 2017-12: [2 2 2 ... 1 1 1]
Period 2017-10: [2 2 2 ... 1 1 1]
Period 2017-08: [2 2 2 ... 0 0 0]
Period 2017-09: [2 2 2 ... 0 1 1]
Period 2017-07: [2 2 2 ... 1 0 1]
Period 2017-05: [2 2 2 ... 0 0 1]
Period 2017-06: [2 2 2 ... 0 1 1]
Period 2017-03: [2 2 2 ... 0 0 1]
Period 2017-04: [2 2 2 ... 1 1 0]
Period 2017-02: [2 2 2 ... 0 0 1]
Period 2017-01: [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 2 2 2 2 2 0 1 2 0 1 1 1 2 1 2 0 0 1 0 1 0
 2 1 1 0 0 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 1
 1 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1
 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0

,recence,frequence,price,freight_value,montant,review_score,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,approval_delay_minutes,approval_delay_hours,approval_delay_days,processing_delay_hours,processing_delay_days,period,Cluster
0,320.833657,22,69.90,33.72,1707.88,1,2017-10-17 13:06:29,2017-10-18 13:06:21,2017-10-22 14:43:54,1439.866667,23.997778,0.0,121.623611,5.0,2017-10,2.0
1,413.616343,21,1.20,7.89,196.17,1,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-31 18:03:02,1445.966667,24.099444,1.0,359.726944,14.0,2017-07,2.0
2,192.733519,20,100.00,10.12,2202.40,1,2018-02-22 15:30:41,2018-02-24 03:20:27,2018-03-05 15:22:27,2149.766667,35.829444,1.0,263.862778,10.0,2018-02,2.0
3,580.473704,15,51.00,1.20,783.00,5,2017-01-30 21:44:49,2017-01-30 22:33:45,2017-02-14 10:48:10,48.933333,0.815556,0.0,349.055833,14.0,2017-01,2.0
4,283.525058,15,65.49,16.22,1225.65,5,2017-11-23 20:30:52,2017-11-24 10:31:10,2017-12-13 20:19:35,840.300000,14.005000,0.0,479.811944,19.0,2017-11,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95827,153.808796,1,93.00,14.01,107.01,4,2018-04-02 13:42:17,2018-04-04 03:10:19,2018-04-13 20:21:08,2248.033333,37.467222,1.0,270.647500,11.0,2018-04,1.0
95828,382.831678,1,149.90,29.45,179.35,5,2017-08-16 13:09:20,2017-08-17 03:10:27,2017-09-13 20:06:02,841.116667,14.018611,0.0,678.945000,28.0,2017-08,0.0
95829,551.916829,1,179.99,15.43,195.42,5,2017-02-28 11:06:43,2017-02-28 11:15:20,2017-03-06 08:57:49,8.616667,0.143611,0.0,141.851667,5.0,2017-02,1.0
95830,413.976678,1,54.90,12.51,67.41,4,2017-07-16 09:40:32,2017-07-16 09:55:12,2017-07-25 18:57:33,14.666667,0.244444,0.0,225.283611,9.0,2017-07,1.0


In [11]:
for period in periods:
    display(clusters_periods[period])

array([2, 2, 2, ..., 0, 1, 0], shape=(7030,), dtype=int32)

array([2, 2, 2, ..., 1, 1, 1], shape=(6835,), dtype=int32)

array([2, 2, 2, ..., 0, 1, 1], shape=(6716,), dtype=int32)

array([0, 0, 0, ..., 1, 1, 1], shape=(6503,), dtype=int32)

array([2, 2, 2, ..., 0, 0, 1], shape=(6471,), dtype=int32)

...

array([2, 2, 2, ..., 0, 0, 1], shape=(2448,), dtype=int32)

array([2, 2, 2, ..., 1, 1, 0], shape=(2213,), dtype=int32)

array([2, 2, 2, ..., 0, 0, 1], shape=(1600,), dtype=int32)

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 0,
       1, 2, 0, 1, 1, 1, 2, 1, 2, 0, 0, 1, 0, 1, 0, 2, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1,

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0], dtype=int32)

In [9]:
from sklearn.metrics import adjusted_rand_score

# Calculer l'ARI entre les clusters de chaque période et la période initiale
reference_period = periods[0]
reference_clusters = clusters_periods[reference_period]

ari_scores = []
for period in periods[1:]:
    try:
        ari = adjusted_rand_score(reference_clusters, clusters_periods[period])
        print(f"ARI={ari}")
        ari_scores.append((period, ari))
    except Exception as e:
        print(f"Une erreur est survenue sur la période {period}: {e}")

# Afficher les scores ARI
for period, ari in ari_scores:
    print(f'Période {period}: ARI = {ari:.2f}')

Une erreur est survenue sur la période 2018-01: Found input variables with inconsistent numbers of samples: [7030, 6835]
Une erreur est survenue sur la période 2018-03: Found input variables with inconsistent numbers of samples: [7030, 6716]
Une erreur est survenue sur la période 2018-04: Found input variables with inconsistent numbers of samples: [7030, 6503]
Une erreur est survenue sur la période 2018-05: Found input variables with inconsistent numbers of samples: [7030, 6471]
Une erreur est survenue sur la période 2018-02: Found input variables with inconsistent numbers of samples: [7030, 6364]
Une erreur est survenue sur la période 2018-08: Found input variables with inconsistent numbers of samples: [7030, 6106]
Une erreur est survenue sur la période 2018-07: Found input variables with inconsistent numbers of samples: [7030, 5862]
Une erreur est survenue sur la période 2018-06: Found input variables with inconsistent numbers of samples: [7030, 5801]
Une erreur est survenue sur la p